In [32]:
import pandas as pd
from pandas import DataFrame
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [33]:
item_categories_df = pd.read_csv("../data/item_categories.csv")
items_df = pd.read_csv('../data/items.csv')
sales_train_df = pd.read_csv("../data/sales_train.csv")
shops_df = pd.read_csv("../data/shops.csv")
test_df = pd.read_csv("../data/test.csv")

In [34]:
sales_train_df.head(3)

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day
0,02.01.2013,0,59,22154,999.0,1.0
1,03.01.2013,0,25,2552,899.0,1.0
2,05.01.2013,0,25,2552,899.0,-1.0


In [35]:
test_df.head(3)

,ID,shop_id,item_id
0,0,5,5037
1,1,5,5320
2,2,5,5233


In [36]:
len(sales_train_df)

2935849

In [37]:
def delete_equal_snop_name(sales_df: DataFrame, shop_df: DataFrame, test_df: DataFrame) -> DataFrame:
    # "Жуковский ул. Чкалова 39м?" and "Жуковский ул. Чкалова 39м²" (correct_id=11, incorrect=10)
    # "Якутск Орджоникидзе, 56" and "!Якутск Орджоникидзе, 56 фран"" (correct_id=57, incorrect=0)
    # "Якутск ТЦ "Центральный"" and "!Якутск ТЦ "Центральный" фран" (correct_id=58, incorrect=1)
    
    equal_shop_map = {
        10: 11,
        0: 57,
        1: 58,
    }
    shop_df = shop_df.copy()
    shop_df['new_shop_id'] = shop_df['shop_id']
    
    # replace equal
    shop_df['new_shop_id'] = shop_df['new_shop_id'].replace(equal_shop_map)
    sales_df['shop_id'] = sales_df['shop_id'].replace(equal_shop_map)
    test_df['shop_id'] = test_df['shop_id'].replace(equal_shop_map)

    # remove equal
    shop_df = shop_df[shop_df['shop_id'] == shop_df['new_shop_id']]

    # drop help column and reset index
    shop_df = shop_df.drop(['new_shop_id'], axis=1)
    shop_df = shop_df.reset_index(drop=True)

    old_new_id_map = dict(zip(shop_df['shop_id'], shop_df.index))

    # replace old ids with new after .reset_index
    sales_df['shop_id'] = sales_df['shop_id'].replace(old_new_id_map)
    test_df['shop_id'] = test_df['shop_id'].replace(old_new_id_map)
    shop_df['shop_id'] = shop_df.index

    
    return sales_df, shop_df, test_df

In [38]:
def remove_outliers(sales_df: DataFrame) -> DataFrame:
    median = sales_df['item_cnt_month'].median()
    sales_without_ones = sales_df[sales_df['item_cnt_month'] != 1]
    MAD = (sales_without_ones['item_cnt_month'] - median).abs().mean() * 1.4826

    sales_df = sales_df[sales_df['item_cnt_month'].between(median - 3*MAD, median + 3*MAD)]
    
    print("MAD: ", median - 3*MAD, median + 3*MAD)
    return sales_df

In [39]:
def move_to_month(sales_df: DataFrame) -> DataFrame:
    sales_df["date"] = pd.to_datetime(sales_df["date"], dayfirst=True)
    sales_df["month"] = sales_df["date"].dt.month
    sales_df = sales_df.groupby(['date_block_num', 'shop_id', 'item_id']).agg({
        'item_cnt_day': 'sum',
        'item_price': 'mean',
        'item_category_id': 'first',
        "month": "first",
    }).reset_index()
    # sales_df = sales_df.drop('item_price', axis=1)
    sales_df.rename(columns={'item_cnt_day': 'item_cnt_month', 'item_price': 'mean_month_price'}, inplace=True)
    
    return sales_df

In [40]:
def fix_data(
    sales_df: DataFrame, 
    items_df: DataFrame, 
    shop_df: DataFrame, 
    test_df: DataFrame,
) -> tuple[DataFrame,]:
    # Check negative price
    # TODO не удалять а заменить на среднюю цену по предмету или нет, у нас 3 млн строк)
    negative_mask = sales_df['item_price'] <= 0
    negative_count = negative_mask.sum()
    if (negative_count > 0):
        print(f"Negative price were find ({negative_count})")
        sales_df = sales_df[~negative_mask]

    # equal shop names
    sales_df, shop_df, test_df = delete_equal_snop_name(sales_df, shop_df, test_df)
    
    # Add category id in data
    sales_df = sales_df.join(items_df['item_category_id'], on='item_id')
    test_df = test_df.join(items_df['item_category_id'], on='item_id')

    # Move to cnt_month
    sales_df = move_to_month(sales_df)

    # Remove outliers
    sales_df = remove_outliers(sales_df)
    
    return sales_df, shop_df, test_df


In [41]:
def create_lag(
    df: pd.DataFrame,
    for_column: str,
    by_column: str,
    new_name: str,
    lag_shift: int = 1,
    insert_NAN_col: bool = True,
) -> pd.DataFrame:
    df_grouped = df.groupby(['date_block_num', for_column])[by_column].mean().reset_index()
    df_grouped['date_block_num'] = df_grouped['date_block_num'] + lag_shift
    df_grouped = df_grouped.rename(columns={by_column: new_name})
    
    df = df.merge(right=df_grouped, how='left', on=['date_block_num', for_column])
    if insert_NAN_col:
        df[new_name+'_NAN'] = df[new_name].isna().astype(int)
    df[new_name] = df[new_name].fillna(0)
    return df

def create_lags(
    df: pd.DataFrame,
    for_columns: list[str],
    by_column: str,
    new_names: list[str],
    lag_shifts: list[int],
    insert_NAN_col: bool = True,
) -> pd.DataFrame:
    for lag_shift in lag_shifts:
        for for_column, new_name in zip(for_columns, new_names):
            df = create_lag(
                df,
                for_column,
                by_column,
                new_name + '_' + str(lag_shift),
                lag_shift,
                insert_NAN_col
            )
    return df

In [42]:
def label_encode_feat(df: pd.DataFrame, columns: list[str], drop=True) -> pd.DataFrame:
    for col in columns:
        df[col+"_encoded"] = LabelEncoder().fit_transform(df[col])
    if drop:
        df = df.drop(columns=columns)
    return df

In [43]:
sales_train_df_fix, shops_df_fix, test_df_fix = fix_data(sales_train_df,
                                                         items_df,
                                                         shops_df,
                                                         test_df,)

Negative price were find (1)
MAD:  -15.553435548674447 17.553435548674447


In [44]:
# Потери в данных
1 - len(sales_train_df_fix)/len(move_to_month(sales_train_df.join(items_df['item_category_id'], on='item_id')))

0.009259075124104843

In [45]:
sales_train_df_fix.head()

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month
0,0,0,27,1.0,2499.0,19,1
1,0,0,33,1.0,499.0,37,1
2,0,0,317,1.0,299.0,45,1
3,0,0,438,1.0,299.0,45,1
4,0,0,471,2.0,399.0,49,1


In [46]:
# take sales only with shop_id that containce in test_data
sales_train_df_fix_uniq = sales_train_df_fix[sales_train_df_fix['shop_id'].isin(test_df_fix['shop_id'].unique())]

# make dataset for future feature extraction
hole_data = pd.concat([sales_train_df_fix_uniq, test_df_fix])
hole_data['date_block_num'] = hole_data['date_block_num'].fillna(34)
hole_data['month'] = hole_data['month'].fillna(11)
hole_data['ID'] = hole_data['ID'].fillna(-1)

int_columns = [
    'date_block_num',
    'month',
    'ID'
]

# make some columns as int to reduce memory щмукруфв
for col in int_columns:
    hole_data[col] = hole_data[col].astype(int)


hole_data

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID
0,0,0,27,1.0,2499.0,19,1,-1
1,0,0,33,1.0,499.0,37,1,-1
2,0,0,317,1.0,299.0,45,1,-1
3,0,0,438,1.0,299.0,45,1,-1
4,0,0,471,2.0,399.0,49,1,-1
...,...,...,...,...,...,...,...,...
214195,34,42,18454,NaN,NaN,55,11,214195
214196,34,42,16188,NaN,NaN,64,11,214196
214197,34,42,15757,NaN,NaN,55,11,214197
214198,34,42,19648,NaN,NaN,40,11,214198


In [47]:
# add shop_city and shop_type features

# fix some incorrect names
shops_df_fix.at[43, 'shop_name'] = 'Сергиев_Посад ТЦ "7Я"'
shops_df_fix.at[17, 'shop_name'] = 'Москва Магазин "Распродажа"'
shops_df_fix.at[7, 'shop_name'] = '- Выездная_Торговля'
shops_df_fix.at[4, 'shop_name'] = 'Воронеж Магазин (Плехановская, 13)'
shops_df_fix.at[52, 'shop_name'] = 'Интернет-магазин Цифровой_склад 1С-Онлайн'
shops_df_fix.at[54, 'shop_name'] = 'Якутск Магазин Орджоникидзе, 56'

shops_df_fix['shop_city'] = shops_df_fix['shop_name'].apply(lambda x: x.split()[0])
shops_df_fix['shop_type'] = shops_df_fix['shop_name'].apply(lambda x: x.split()[1])

shops_df_fix = label_encode_feat(shops_df_fix, ['shop_city', 'shop_type'])

shops_df_fix

,shop_name,shop_id,shop_city_encoded,shop_type_encoded
0,"Адыгея ТЦ ""Мега""",0,1,6
1,"Балашиха ТРК ""Октябрь-Киномир""",1,2,4
2,"Волжский ТЦ ""Волга Молл""",2,3,6
3,"Вологда ТРЦ ""Мармелад""",3,4,5
4,"Воронеж Магазин (Плехановская, 13)",4,5,2
5,"Воронеж ТРЦ ""Максимир""",5,5,5
6,"Воронеж ТРЦ Сити-Парк ""Град""",6,5,5
7,- Выездная_Торговля,7,0,0
8,Жуковский ул. Чкалова 39м²,8,6,9
9,Интернет-магазин ЧС,9,7,8


In [48]:
# add shop_city and shop_type in main dataset
hole_data_shp = hole_data.merge(shops_df_fix, how='left', on='shop_id').drop(columns='shop_name')

In [49]:
# make new feature sub_category for item_category
item_categories_df['sub_category'] = item_categories_df['item_category_name'].apply(lambda x: x.split(' - ')[0])
item_categories_df = label_encode_feat(item_categories_df, ['sub_category'], drop=False)

item_categories_df

,item_category_name,item_category_id,sub_category,sub_category_encoded
0,PC - Гарнитуры/Наушники,0,PC,0
1,Аксессуары - PS2,1,Аксессуары,1
2,Аксессуары - PS3,2,Аксессуары,1
3,Аксессуары - PS4,3,Аксессуары,1
4,Аксессуары - PSP,4,Аксессуары,1
...,...,...,...,...
79,Служебные,79,Служебные,16
80,Служебные - Билеты,80,Служебные,16
81,Чистые носители (шпиль),81,Чистые носители (шпиль),17
82,Чистые носители (штучные),82,Чистые носители (штучные),18


In [50]:
# add sub_category feature in main dataset
hole_data_cat = hole_data_shp.merge(item_categories_df, how='left', on='item_category_id').drop(columns='item_category_name')
hole_data_cat = hole_data_cat.drop(columns='sub_category')

hole_data_cat

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,sub_category_encoded
0,0,0,27,1.0,2499.0,19,1,-1,1,6,5
1,0,0,33,1.0,499.0,37,1,-1,1,6,11
2,0,0,317,1.0,299.0,45,1,-1,1,6,12
3,0,0,438,1.0,299.0,45,1,-1,1,6,12
4,0,0,471,2.0,399.0,49,1,-1,1,6,12
...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,13
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,14
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,13
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,11


In [51]:
# add lags for columns
hole_data_new = create_lags(
    hole_data_cat,
    ['item_id', 'item_category_id', 'shop_city_encoded', 'shop_type_encoded', 'sub_category_encoded', 'shop_id'],
    'mean_month_price', # lag by this column
    ['item_price_lag', 'category_price_lag', 'shop_city_price_lag', 'shop_type_price_lag', 'sub_category_price_lag', 'shop_price_lag'],
    lag_shifts=[1, 2, 3],
    # insert_NAN_col=False
)
hole_data_new = create_lags(
    hole_data_new,
    ['item_id', 'item_category_id', 'shop_city_encoded', 'shop_type_encoded', 'sub_category_encoded', 'shop_id'],
    'item_cnt_month', # lag by this column
    ['item_cnt_lag', 'category_cnt_lag', 'shop_city_cnt_lag', 'shop_type_cnt_lag', 'sub_category_cnt_lag', 'shop_cnt_lag'],
    lag_shifts=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    insert_NAN_col=False
)

# make month_since_first_sale
first_sale_month = hole_data_new[hole_data_new['item_price_lag_1_NAN'] == 1] \
    .groupby(['item_id'])['date_block_num'] \
    .first() \
    .rename('months_since_first_sale') \
    .reset_index()
hole_data_new = hole_data_new.merge(first_sale_month, how='left', on=['item_id'])
hole_data_new['months_since_first_sale'] = hole_data_new['date_block_num'] - hole_data_new['months_since_first_sale']

hole_data_new

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,shop_type_cnt_lag_11,sub_category_cnt_lag_11,shop_cnt_lag_11,item_cnt_lag_12,category_cnt_lag_12,shop_city_cnt_lag_12,shop_type_cnt_lag_12,sub_category_cnt_lag_12,shop_cnt_lag_12,months_since_first_sale
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,2.26502,1.578208,1.709114,0.0,1.427743,1.572549,1.945795,1.436895,1.588525,11
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,2.26502,2.201372,1.709114,0.0,1.586687,1.572549,1.945795,1.772190,1.588525,2
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,2.26502,1.578208,1.709114,1.0,1.427743,1.572549,1.945795,1.436895,1.588525,34
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,2.26502,1.821388,1.709114,0.0,1.705631,1.572549,1.945795,1.616361,1.588525,11


In [52]:
# drop NAN columns (it proved useless)
for column in hole_data_new.columns:
    if '_NAN' in column:
        hole_data_new = hole_data_new.drop(columns=[column])
hole_data_new

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,shop_type_cnt_lag_11,sub_category_cnt_lag_11,shop_cnt_lag_11,item_cnt_lag_12,category_cnt_lag_12,shop_city_cnt_lag_12,shop_type_cnt_lag_12,sub_category_cnt_lag_12,shop_cnt_lag_12,months_since_first_sale
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.00000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,2.26502,1.578208,1.709114,0.0,1.427743,1.572549,1.945795,1.436895,1.588525,11
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,2.26502,2.201372,1.709114,0.0,1.586687,1.572549,1.945795,1.772190,1.588525,2
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,2.26502,1.578208,1.709114,1.0,1.427743,1.572549,1.945795,1.436895,1.588525,34
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,2.26502,1.821388,1.709114,0.0,1.705631,1.572549,1.945795,1.616361,1.588525,11


In [53]:
# make trend features
def make_trend_from_lags(
    df: pd.DataFrame,
    lag_name: str,
    new_name: str,
    lag_pairs: list[tuple[int, int]]
) -> pd.DataFrame:
    for first, second in lag_pairs:
        df[new_name+'_'+str(first)+'_'+str(second)] = df[lag_name+'_'+str(first)] - df[lag_name+'_'+str(second)]
    return df

def make_trends(
    df: pd.DataFrame,
    cols: list[str],
    lag_pairs: list[tuple[int, int]],
) -> pd.DataFrame:
    df_new = pd.DataFrame()
    for col in cols:
        new_name = col.rsplit('_', 1)[0]  + '_trend'
        df_new = make_trend_from_lags(
            df,
            col,
            new_name,
            lag_pairs,
        )
    return df_new

# make trend features (lag_1 - lag_2 and lag_1 - lag_12) for above columns
hole_data_trends = make_trends(
    hole_data_new,
    ['item_cnt_lag', 'item_price_lag', 'sub_category_price_lag', 'sub_category_cnt_lag', 'category_cnt_lag', 'shop_city_cnt_lag', 'shop_type_cnt_lag'],
    [(1,2)]
)
hole_data_trends = make_trends(
    hole_data_new,
    ['item_cnt_lag', 'sub_category_cnt_lag', 'category_cnt_lag', 'shop_city_cnt_lag', 'shop_type_cnt_lag'],
    [(1,12)]
)


hole_data_trends

C:\Users\vaveyko\AppData\Local\Temp\ipykernel_20568\3316112113.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_name+'_'+str(first)+'_'+str(second)] = df[lag_name+'_'+str(first)] - df[lag_name+'_'+str(second)]
C:\Users\vaveyko\AppData\Local\Temp\ipykernel_20568\3316112113.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[new_name+'_'+str(first)+'_'+str(second)] = df[lag_name+'_'+str(first)] - df[lag_name+'_'+str(second)]


,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,sub_category_price_trend_1_2,sub_category_cnt_trend_1_2,category_cnt_trend_1_2,shop_city_cnt_trend_1_2,shop_type_cnt_trend_1_2,item_cnt_trend_1_12,sub_category_cnt_trend_1_12,category_cnt_trend_1_12,shop_city_cnt_trend_1_12,shop_type_cnt_trend_1_12
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,0.000000,0.000000,0.000000
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,0.000000,0.000000,0.000000
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,0.000000,0.000000,0.000000
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,0.000000,0.000000,0.000000
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.0000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,-1.737156,0.054004,0.092660,-0.105688,-0.0449,1.0,-0.115856,-0.086886,-0.184209,-0.252874
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,54.720428,-0.054449,0.044865,-0.105688,-0.0449,1.0,-0.232606,-0.262528,-0.184209,-0.252874
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,-1.737156,0.054004,0.092660,-0.105688,-0.0449,0.0,-0.115856,-0.086886,-0.184209,-0.252874
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,-0.737427,0.080190,0.061594,-0.105688,-0.0449,1.0,-0.044879,-0.118329,-0.184209,-0.252874


In [54]:
def ratio_lag_mean_1_3_to_lag_1(
    df: pd.DataFrame,
    cols: list[str],
) -> pd.DataFrame:
    new_cols = {}
    for col in cols:
        mean_col = col + '_mean_1_3'
        ratio_col = 'ratio_' + col + '_mean_1_3_to_lag1'
        mean_vals = df[[col + '_1', col + '_2', col + '_3']].mean(axis=1)
        ratio_vals = mean_vals / (df[col + '_1'] + 1e-6)

        new_cols[mean_col] = mean_vals
        new_cols[ratio_col] = ratio_vals
    new_df = pd.DataFrame(new_cols, index=df.index)
    return pd.concat([df, new_df], axis=1)

# make feature ratio lag mean to the lag_1
hole_data_w_roll = ratio_lag_mean_1_3_to_lag_1(
    hole_data_trends,
    ['item_cnt_lag', 'item_price_lag', 'sub_category_cnt_lag', 'category_cnt_lag', 'shop_city_cnt_lag', 'shop_type_cnt_lag']
)
hole_data_w_roll

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,item_price_lag_mean_1_3,ratio_item_price_lag_mean_1_3_to_lag1,sub_category_cnt_lag_mean_1_3,ratio_sub_category_cnt_lag_mean_1_3_to_lag1,category_cnt_lag_mean_1_3,ratio_category_cnt_lag_mean_1_3_to_lag1,shop_city_cnt_lag_mean_1_3,ratio_shop_city_cnt_lag_mean_1_3_to_lag1,shop_type_cnt_lag_mean_1_3,ratio_shop_type_cnt_lag_mean_1_3_to_lag1
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,99.000000,1.000000,1.286211,0.973636,1.291330,0.963063,1.417012,1.020651,1.733881,1.024195
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,860.655556,0.633301,1.579365,1.025837,1.337534,1.010100,1.417012,1.020651,1.733881,1.024195
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,229.000000,1.000000,1.286211,0.973636,1.291330,0.963063,1.417012,1.020651,1.733881,1.024195
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,95.700000,1.074074,1.524891,0.970352,1.567541,0.987550,1.417012,1.020651,1.733881,1.024195


In [55]:
# remove reduntant columns lags for below columns from 4 to 11
delete_lags = ['item_cnt_lag', 'category_cnt_lag', 'shop_city_cnt_lag', 'shop_type_cnt_lag', 'sub_category_cnt_lag', 'shop_cnt_lag']
to_delete = []
for col in delete_lags:
    for i in range(4, 12):
        to_delete.append(col+'_'+str(i))
hole_data_w_roll = hole_data_w_roll.drop(columns=to_delete)
hole_data_w_roll

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,item_price_lag_mean_1_3,ratio_item_price_lag_mean_1_3_to_lag1,sub_category_cnt_lag_mean_1_3,ratio_sub_category_cnt_lag_mean_1_3_to_lag1,category_cnt_lag_mean_1_3,ratio_category_cnt_lag_mean_1_3_to_lag1,shop_city_cnt_lag_mean_1_3,ratio_shop_city_cnt_lag_mean_1_3_to_lag1,shop_type_cnt_lag_mean_1_3,ratio_shop_type_cnt_lag_mean_1_3_to_lag1
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,99.000000,1.000000,1.286211,0.973636,1.291330,0.963063,1.417012,1.020651,1.733881,1.024195
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,860.655556,0.633301,1.579365,1.025837,1.337534,1.010100,1.417012,1.020651,1.733881,1.024195
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,229.000000,1.000000,1.286211,0.973636,1.291330,0.963063,1.417012,1.020651,1.733881,1.024195
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,95.700000,1.074074,1.524891,0.970352,1.567541,0.987550,1.417012,1.020651,1.733881,1.024195


In [56]:
# add revenue features

hole_data_revenue = hole_data_w_roll
hole_data_revenue['item_revenue_lag_1'] = hole_data_revenue['item_cnt_lag_1'] * hole_data_revenue['item_price_lag_1']
hole_data_revenue['item_revenue_lag_2'] = hole_data_revenue['item_cnt_lag_2'] * hole_data_revenue['item_price_lag_2']
hole_data_revenue['shop_revenue_lag_1'] = hole_data_revenue['shop_cnt_lag_1'] * hole_data_revenue['shop_price_lag_1']


hole_data_revenue['shop_revenue_lag_2'] = hole_data_revenue['shop_cnt_lag_2'] * hole_data_revenue['shop_price_lag_2'] 
hole_data_revenue = make_trends(hole_data_revenue, ['shop_revenue_lag', 'item_revenue_lag'], [(1, 2)])


# add item_cnt_sales percent of shop sales
hole_data_revenue['item_percent_of_shop_cnt'] = hole_data_revenue['item_cnt_lag_1'] / (hole_data_revenue['shop_cnt_lag_1']+1e-4)

hole_data_revenue

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,ratio_shop_city_cnt_lag_mean_1_3_to_lag1,shop_type_cnt_lag_mean_1_3,ratio_shop_type_cnt_lag_mean_1_3_to_lag1,item_revenue_lag_1,item_revenue_lag_2,shop_revenue_lag_1,shop_revenue_lag_2,shop_revenue_trend_1_2,item_revenue_trend_1_2,item_percent_of_shop_cnt
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,1.020651,1.733881,1.024195,99.0,99.000000,1533.356731,1944.671427,-411.314696,0.000000,0.739562
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,1.020651,1.733881,1.024195,1359.0,1222.966667,1533.356731,1944.671427,-411.314696,136.033333,0.739562
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,1.020651,1.733881,1.024195,229.0,229.000000,1533.356731,1944.671427,-411.314696,0.000000,0.739562
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,1.020651,1.733881,1.024195,89.1,99.000000,1533.356731,1944.671427,-411.314696,-9.900000,0.739562


In [57]:
# make pair features
def make_collab_lag(
    df: DataFrame,
    names: list[list[str]],
    new_names: list[str],
    lags: list[int],
) -> pd.DataFrame:
    new_cols = {}

    for new_name, group in zip(new_names, names):
        group = df.groupby(group)['item_cnt_month']
        for lag in lags:
            col_name = f'{new_name}_{lag}'
            feature = group.shift(lag).fillna(0)
            new_cols[col_name] = feature

    new_df = pd.DataFrame(new_cols, index=df.index)
    return pd.concat([df, new_df], axis=1)


df = hole_data_revenue.sort_values(['shop_id', 'item_id', 'date_block_num'])

# make shop item cnt lag
# make shop sub_category cnt lag
# make sub_category item cnt lag
# make shop-type item cnt lag
# make shop-city item cnt lag
new_names = [
        'item_shop_cnt_month_lag',
        'sub-category_shop_cnt_month_lag',
        'sub-category_item_cnt_month_lag',
        'item_shop-type_cnt_month_lag',
        'item_shop-city_cnt_month_lag'
    ]
df = make_collab_lag(
    df,
    names = [
        ['shop_id', 'item_id'],
        ['shop_id', 'sub_category_encoded'],
        ['sub_category_encoded', 'item_id'],
        ['shop_type_encoded', 'item_id'],
        ['shop_city_encoded', 'item_id']
        
    ],
    new_names = new_names,
    lags=[1, 2, 3]
)

df = df.sort_index()

# make trends for above features
df = make_trends(
    df,
    new_names,
    [(1, 2)]
)

# make ratio for above features
df = ratio_lag_mean_1_3_to_lag_1(
    df,
    new_names
)

df

,date_block_num,shop_id,item_id,item_cnt_month,mean_month_price,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,...,item_shop_cnt_month_lag_mean_1_3,ratio_item_shop_cnt_month_lag_mean_1_3_to_lag1,sub-category_shop_cnt_month_lag_mean_1_3,ratio_sub-category_shop_cnt_month_lag_mean_1_3_to_lag1,sub-category_item_cnt_month_lag_mean_1_3,ratio_sub-category_item_cnt_month_lag_mean_1_3_to_lag1,item_shop-type_cnt_month_lag_mean_1_3,ratio_item_shop-type_cnt_month_lag_mean_1_3_to_lag1,item_shop-city_cnt_month_lag_mean_1_3,ratio_item_shop-city_cnt_month_lag_mean_1_3_to_lag1
0,0,0,27,1.0,2499.0,19,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,0,33,1.0,499.0,37,1,-1,1,6,...,0.000000,0.000000,0.666667,666666.666667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,0,317,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,0,438,1.0,299.0,45,1,-1,1,6,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,0,471,2.0,399.0,49,1,-1,1,6,...,0.000000,0.000000,1.000000,1000000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,34,42,18454,NaN,NaN,55,11,214195,20,6,...,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999
1534425,34,42,16188,NaN,NaN,64,11,214196,20,6,...,0.000000,0.000000,0.333333,333333.333333,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1534426,34,42,15757,NaN,NaN,55,11,214197,20,6,...,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999
1534427,34,42,19648,NaN,NaN,40,11,214198,20,6,...,0.000000,0.000000,0.666667,0.666666,0.333333,333333.333333,0.333333,333333.333333,0.333333,333333.333333


In [58]:
# divide data on train and test
sales_train_df_fix_featured = df[df['ID'] == -1]
sales_train_df_fix_featured = sales_train_df_fix_featured.drop(columns=['ID', 'mean_month_price'])
sales_train_df_fix_featured

,date_block_num,shop_id,item_id,item_cnt_month,item_category_id,month,shop_city_encoded,shop_type_encoded,sub_category_encoded,item_price_lag_1,...,item_shop_cnt_month_lag_mean_1_3,ratio_item_shop_cnt_month_lag_mean_1_3_to_lag1,sub-category_shop_cnt_month_lag_mean_1_3,ratio_sub-category_shop_cnt_month_lag_mean_1_3_to_lag1,sub-category_item_cnt_month_lag_mean_1_3,ratio_sub-category_item_cnt_month_lag_mean_1_3_to_lag1,item_shop-type_cnt_month_lag_mean_1_3,ratio_item_shop-type_cnt_month_lag_mean_1_3_to_lag1,item_shop-city_cnt_month_lag_mean_1_3,ratio_item_shop-city_cnt_month_lag_mean_1_3_to_lag1
0,0,0,27,1.0,19,1,1,6,5,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0,0,33,1.0,37,1,1,6,11,0.000000,...,0.000000,0.000000,0.666667,666666.666667,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0,0,317,1.0,45,1,1,6,12,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0,0,438,1.0,45,1,1,6,12,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,0,0,471,2.0,49,1,1,6,12,0.000000,...,0.000000,0.000000,1.000000,1000000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1320224,33,56,22087,6.0,83,10,29,6,19,116.992800,...,3.333333,1.111111,3.333333,1.111111,3.333333,1.111111,3.333333,1.111111,3.333333,1.111111
1320225,33,56,22088,2.0,83,10,29,6,19,117.272727,...,5.000000,4.999995,5.000000,4.999995,5.000000,4.999995,5.000000,4.999995,5.000000,4.999995
1320226,33,56,22091,1.0,83,10,29,6,19,178.142857,...,1.666667,0.555555,1.666667,0.555555,1.666667,0.555555,1.666667,0.555555,1.666667,0.555555
1320227,33,56,22100,1.0,42,10,29,6,12,620.580645,...,0.333333,0.333333,0.666667,0.666666,2.666667,2.666664,2.666667,2.666664,0.333333,0.333333


In [59]:
test_df_fix_featured = df[df['date_block_num'] == 34]
test_df_fix_featured = test_df_fix_featured.drop(columns=['item_cnt_month', 'date_block_num', 'mean_month_price'])
test_df_fix_featured

,shop_id,item_id,item_category_id,month,ID,shop_city_encoded,shop_type_encoded,sub_category_encoded,item_price_lag_1,category_price_lag_1,...,item_shop_cnt_month_lag_mean_1_3,ratio_item_shop_cnt_month_lag_mean_1_3_to_lag1,sub-category_shop_cnt_month_lag_mean_1_3,ratio_sub-category_shop_cnt_month_lag_mean_1_3_to_lag1,sub-category_item_cnt_month_lag_mean_1_3,ratio_sub-category_item_cnt_month_lag_mean_1_3_to_lag1,item_shop-type_cnt_month_lag_mean_1_3,ratio_item_shop-type_cnt_month_lag_mean_1_3_to_lag1,item_shop-city_cnt_month_lag_mean_1_3,ratio_item_shop-city_cnt_month_lag_mean_1_3_to_lag1
1320229,3,5037,19,11,0,4,5,5,1499.000000,1537.237267,...,1.666667,1.666665,1.666667,1.666665e+00,1.666667,1.666665,1.666667,1.666665,1.666667,1.666665
1320230,3,5320,55,11,1,4,5,13,0.000000,327.477705,...,0.000000,0.000000,1.666667,1.666667e+06,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1320231,3,5233,19,11,2,4,5,5,1199.000000,1537.237267,...,1.666667,1.666665,1.666667,1.666665e+00,1.666667,1.666665,1.666667,1.666665,1.666667,1.666665
1320232,3,5232,23,11,3,4,5,5,1185.473684,1631.173151,...,0.333333,0.333333,1.000000,9.999990e-01,0.666667,0.666666,0.333333,0.333333,0.333333,0.333333
1320233,3,5268,20,11,4,4,5,5,0.000000,3154.375844,...,0.000000,0.000000,2.000000,6.666664e-01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1534424,42,18454,55,11,214195,20,6,13,99.000000,327.477705,...,1.000000,0.999999,1.000000,9.999990e-01,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999
1534425,42,16188,64,11,214196,20,6,14,1359.000000,1254.647375,...,0.000000,0.000000,0.333333,3.333333e+05,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1534426,42,15757,55,11,214197,20,6,13,229.000000,327.477705,...,1.000000,0.999999,1.000000,9.999990e-01,1.000000,0.999999,1.000000,0.999999,1.000000,0.999999
1534427,42,19648,40,11,214198,20,6,11,89.100000,259.167772,...,0.000000,0.000000,0.666667,6.666660e-01,0.333333,333333.333333,0.333333,333333.333333,0.333333,333333.333333


In [60]:
# save in files
shops_df_fix.to_csv("../data/preprocessed/shops_preprocessed.csv", index=False)
test_df_fix_featured.to_csv("../data/preprocessed/test_preprocessed.csv", index=False)
# sales_train_df_fix_featured.to_csv("../data/preprocessed/sales_train_preprocessed_long.csv", index=False)
sales_train_df_fix_featured.to_csv("../data/preprocessed/sales_train_preprocessed.csv", index=False)